# SignBridge v6 ULTRA - AUTSL 226 Sinif (95%+ Hedef)

**Yeni teknikler:**
- Velocity + Acceleration + Inter-landmark distances (675 + ~100 distance feat)
- 3 fazli egitim (freeze -> partial -> full)
- Focal Loss (zor siniflara odaklanma)
- SWA (Stochastic Weight Averaging, son 20 epoch)
- TTA (Test-Time Augmentation, 7 varyant)
- R-Drop regularization
- 150 epoch, patience=30
- Mixup KAPALI, Label smoothing 0.02

**Runtime > Run all > Drive izni ver > ~3-4 saat bekle**

In [ ]:
#@title 1 - KURULUM + DRIVE
import subprocess, sys, os, shutil

for pkg in ['torch', 'torchvision', 'numpy', 'tqdm']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Paketler hazir')

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print('Drive bagli')

SAVE_DIR = '/content/drive/MyDrive/AUTSL_Proje/Models_v6_ultra'
if os.path.exists(SAVE_DIR):
    for f in os.listdir(SAVE_DIR):
        if f.endswith('.pt'):
            os.remove(os.path.join(SAVE_DIR, f))
            print(f'Silindi: {f}')
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Kayit: {SAVE_DIR}')

In [ ]:
#@title 2 - KONFIGURASYON + VERI YUKLEME
import numpy as np, json, csv, math, time, random, gc, copy
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

class CONFIG:
    PROJE_DIR = '/content/drive/MyDrive/AUTSL_Proje'
    KOORDINAT_DIR = f'{PROJE_DIR}/Koordinatlar'
    CLASS_LIST_CSV = f'{PROJE_DIR}/SignList_ClassId_TR_EN (1).csv'
    SAVE_DIR = '/content/drive/MyDrive/AUTSL_Proje/Models_v6_ultra'
    TRAIN_NPZ = f'{KOORDINAT_DIR}/all_train_packed.npz'
    VAL_NPZ = f'{KOORDINAT_DIR}/all_val_packed.npz'
    TEST_NPZ = f'{KOORDINAT_DIR}/all_test_packed.npz'
    # Model architecture (transformer layers match pretrained)
    RAW_INPUT = 225  # MediaPipe: 75 landmarks x 3
    D_MODEL = 384; NHEAD = 12; NUM_LAYERS = 6
    NUM_CLASSES = 226; DROPOUT = 0.25  # Dusuruldu (0.35->0.25)
    SEQ_LENGTH = 30
    # Features: raw(225) + velocity(225) + accel(225) + distances(~50 pair)
    USE_VELOCITY = True
    USE_ACCELERATION = True
    USE_DISTANCES = True
    # Training
    EPOCHS = 150; BATCH_SIZE = 64
    LR = 8e-6  # Cok dusuk
    WEIGHT_DECAY = 0.005
    WARMUP_EPOCHS = 5
    LABEL_SMOOTHING = 0.02
    MIXUP_ALPHA = 0.0  # KAPALI
    GRADIENT_CLIP = 1.0; EMA_DECAY = 0.999  # Daha yavas EMA
    # Augmentation (hafif)
    AUG_NOISE_STD = 0.003
    AUG_MIRROR_PROB = 0.2
    AUG_SPEED_RANGE = (0.9, 1.1)
    AUG_SCALE_RANGE = (0.95, 1.05)
    AUG_DROPOUT_PROB = 0.03
    AUG_TIME_MASK_PROB = 0.15  # Temporal masking
    AUG_TIME_MASK_MAX = 3  # Max 3 frame mask
    PATIENCE = 30
    # Phase training
    PHASE1_EPOCHS = 8   # input_conv + classifier only
    PHASE2_EPOCHS = 25  # + son 3 transformer layer
    # Phase3: full (kalan epochlar)
    # SWA
    SWA_START_EPOCH = 80  # SWA baslangiC
    SWA_LR = 2e-6
    # Focal loss
    FOCAL_GAMMA = 2.0
    # R-Drop
    RDROP_ALPHA = 0.5  # KL divergence weight
    # TTA
    TTA_VARIANTS = 7

cfg = CONFIG()

# Sinif isimleri
class_names = {}
if os.path.exists(cfg.CLASS_LIST_CSV):
    with open(cfg.CLASS_LIST_CSV, 'r', encoding='utf-8') as f:
        reader = csv.reader(f); next(reader, None)
        for row in reader:
            if len(row) >= 2:
                try: class_names[int(row[0])] = row[1].strip()
                except: pass
    print(f'{len(class_names)} sinif ismi yuklendi')

# Veri yukle
data = {}
for split, fpath in [('train', cfg.TRAIN_NPZ), ('val', cfg.VAL_NPZ), ('test', cfg.TEST_NPZ)]:
    if not os.path.exists(fpath):
        print(f'HATA {split} bulunamadi: {fpath}'); continue
    d = np.load(fpath); keys = list(d.keys())
    X = d['x'].astype(np.float32) if 'x' in keys else d[keys[0]].astype(np.float32)
    y = d['y'].astype(np.int64) if 'y' in keys else d[keys[1]].astype(np.int64)
    data[split] = (X, y)
    print(f'{split:5s}: {X.shape[0]:6d} ornek | shape={X.shape}')

X_train, y_train = data['train']
X_val, y_val = data['val']
X_test, y_test = data['test']

# Sinif dagilimi
class_counts = Counter(y_train.tolist())
max_count = max(class_counts.values())
class_weights = torch.tensor(
    [(max_count / class_counts.get(i, max_count))**0.5 for i in range(cfg.NUM_CLASSES)],
    dtype=torch.float32
)
print(f'Sinif agirliklari: min={class_weights.min():.2f}, max={class_weights.max():.2f}')
print(f'Veri yuklendi')

In [ ]:
#@title 3 - OZELLIK MUHENDISLIGI + MODEL + DATASET

# ===== INTER-LANDMARK DISTANCE PAIRS =====
# MediaPipe: 0-10 pose(11 lm), 11-31 face(21 lm), 32-52 left_hand(21 lm), 53-74 right_hand(21 lm)
# Key distances that capture sign language structure
DISTANCE_PAIRS = [
    # Hand to face distances (critical for signs near face)
    (32, 11), (32, 14), (32, 18),  # left wrist -> nose, left eye, right eye
    (53, 11), (53, 14), (53, 18),  # right wrist -> nose, left eye, right eye
    (36, 11), (36, 14),  # left index tip -> nose, left eye
    (57, 11), (57, 18),  # right index tip -> nose, right eye
    (40, 11), (61, 11),  # left middle tip, right middle tip -> nose
    # Hand to hand distances
    (32, 53),  # left wrist -> right wrist
    (36, 57),  # left index -> right index
    (40, 61),  # left middle -> right middle
    (44, 65),  # left ring -> right ring
    (48, 69),  # left pinky -> right pinky
    # Hand internal (finger spread)
    (36, 48), (57, 69),  # index-pinky distance each hand
    (36, 44), (57, 65),  # index-ring each hand
    (33, 36), (54, 57),  # thumb tip - index tip each hand
    (33, 40), (54, 61),  # thumb tip - middle tip each hand
    # Hand to shoulder/hip (relative position)
    (32, 0), (53, 0),   # wrists to body center (pose 0)
    (32, 2), (53, 2),   # wrists to shoulders
    (32, 3), (53, 3),
    # Finger tip to wrist (hand openness)
    (36, 32), (40, 32), (44, 32), (48, 32),  # left hand
    (57, 53), (61, 53), (65, 53), (69, 53),  # right hand
    # Cross hand-body
    (36, 0), (57, 0),  # index tips to body center
    (32, 6), (53, 6),  # wrists to hip area
]
NUM_DISTANCES = len(DISTANCE_PAIRS)
print(f'Distance pairs: {NUM_DISTANCES}')

def compute_distances(seq):
    """seq: (T, 225) -> (T, NUM_DISTANCES)"""
    T = seq.shape[0]
    dists = np.zeros((T, NUM_DISTANCES), dtype=np.float32)
    for i, (a, b) in enumerate(DISTANCE_PAIRS):
        if a*3+2 < seq.shape[1] and b*3+2 < seq.shape[1]:
            ax, ay = seq[:, a*3], seq[:, a*3+1]
            bx, by = seq[:, b*3], seq[:, b*3+1]
            dists[:, i] = np.sqrt((ax-bx)**2 + (ay-by)**2 + 1e-8)
    return dists

def build_features(sequence, cfg):
    """Build comprehensive feature vector."""
    parts = [sequence]  # raw 225
    if cfg.USE_VELOCITY:
        v = np.zeros_like(sequence)
        v[1:] = sequence[1:] - sequence[:-1]
        parts.append(v)
    if cfg.USE_ACCELERATION:
        a = np.zeros_like(sequence)
        a[2:] = sequence[2:] - 2*sequence[1:-1] + sequence[:-2]
        parts.append(a)
    if cfg.USE_DISTANCES:
        d = compute_distances(sequence)
        parts.append(d)
    return np.concatenate(parts, axis=-1).astype(np.float32)

def compute_feature_size(cfg):
    size = cfg.RAW_INPUT
    if cfg.USE_VELOCITY: size += cfg.RAW_INPUT
    if cfg.USE_ACCELERATION: size += cfg.RAW_INPUT
    if cfg.USE_DISTANCES: size += NUM_DISTANCES
    return size

FEATURE_SIZE = compute_feature_size(cfg)
print(f'Feature size: {FEATURE_SIZE} (raw=225 + vel=225 + acc=225 + dist={NUM_DISTANCES})')

# ===== AUGMENTATION =====
def augment_sequence(seq, cfg):
    aug = seq.copy()
    T = aug.shape[0]
    # Speed perturbation
    if random.random() < 0.35:
        new_T = max(10, int(T * random.uniform(*cfg.AUG_SPEED_RANGE)))
        idx = np.clip(np.linspace(0, T-1, new_T).astype(int), 0, T-1)
        aug = aug[idx]
    # Temporal shift
    if random.random() < 0.15:
        aug = np.roll(aug, random.randint(-2, 2), axis=0)
    # Gaussian noise
    if random.random() < 0.35:
        aug = aug + np.random.randn(*aug.shape).astype(np.float32) * cfg.AUG_NOISE_STD
    # Mirror (left-right swap)
    if random.random() < cfg.AUG_MIRROR_PROB:
        m = aug.copy()
        for i in range(0, min(225, aug.shape[1]), 3): m[:, i] = 1.0 - m[:, i]
        if aug.shape[1] >= 225:
            lh, rh = m[:, 99:162].copy(), m[:, 162:225].copy()
            m[:, 99:162], m[:, 162:225] = rh, lh
        aug = m
    # Scale
    if random.random() < 0.15:
        s = random.uniform(*cfg.AUG_SCALE_RANGE)
        for i in range(0, min(225, aug.shape[1]), 3): aug[:, i] *= s; aug[:, i+1] *= s
    # Landmark dropout
    if random.random() < cfg.AUG_DROPOUT_PROB:
        n_lm = min(75, aug.shape[1] // 3)
        for li in random.sample(range(n_lm), max(1, int(n_lm * 0.03))):
            aug[:, li*3:li*3+3] = 0.0
    # Temporal masking (mask consecutive frames)
    if random.random() < cfg.AUG_TIME_MASK_PROB:
        T_cur = aug.shape[0]
        mask_len = random.randint(1, cfg.AUG_TIME_MASK_MAX)
        start = random.randint(0, max(0, T_cur - mask_len))
        # Replace with interpolation instead of zeros
        if start > 0 and start + mask_len < T_cur:
            for t in range(start, min(start + mask_len, T_cur)):
                alpha = (t - start + 1) / (mask_len + 1)
                aug[t] = aug[start-1] * (1-alpha) + aug[min(start+mask_len, T_cur-1)] * alpha
    return aug

def pad_or_truncate(sequence, target_length):
    T = len(sequence)
    if T == target_length: return sequence
    elif T > target_length:
        s = (T - target_length) // 2; return sequence[s:s + target_length]
    else:
        return np.concatenate([sequence, np.tile(sequence[-1:], (target_length - T, 1))], axis=0)

class AUTSLDataset(Dataset):
    def __init__(self, X, y, cfg, is_train=True):
        self.X, self.y, self.cfg, self.is_train = X, y, cfg, is_train
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        seq = self.X[idx].copy()
        if self.is_train: seq = augment_sequence(seq, self.cfg)
        seq = pad_or_truncate(seq, self.cfg.SEQ_LENGTH)
        seq = build_features(seq, self.cfg)
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# ===== MODEL =====
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class SignTransformerPro(nn.Module):
    def __init__(self, input_size, d_model=384, nhead=12, num_layers=6, num_classes=226, dropout=0.35):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Linear(input_size, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.conv_block = nn.Sequential(
            nn.Conv1d(d_model, d_model, 3, padding=1, groups=d_model),
            nn.Conv1d(d_model, d_model, 1),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model//4), nn.Tanh(), nn.Linear(d_model//4, 1))
            for _ in range(4)])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model*5), nn.Dropout(dropout),
            nn.Linear(d_model*5, d_model*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model*2, d_model), nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(d_model, num_classes))
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.input_conv(x)
        x = x + self.conv_block(x.transpose(1,2)).transpose(1,2)
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], dim=1)
        x = self.transformer(self.pos_encoder(x))
        seq = x[:, 1:]
        pooled = [F.softmax(h(seq), dim=1) * seq for h in self.pool_heads]
        pooled = [p.sum(dim=1) for p in pooled]
        return self.classifier(torch.cat(pooled + [seq.mean(dim=1)], dim=1))

# ===== FOCAL LOSS =====
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing
    def forward(self, logits, targets):
        # Standard CE with label smoothing
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction='none')
        # Focal modulation
        pt = torch.exp(-ce)  # probability of correct class
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()

# ===== EMA =====
class EMAModel:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            if param.requires_grad: self.shadow[name] = param.data.clone()
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data
    def apply(self, model):
        backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                backup[name] = param.data.clone(); param.data = self.shadow[name]
        return backup
    def restore(self, model, backup):
        for name, param in model.named_parameters():
            if param.requires_grad and name in backup: param.data = backup[name]
    def state_dict(self): return dict(self.shadow)
    def load_state_dict(self, sd):
        for k in sd:
            if k in self.shadow: self.shadow[k] = sd[k].clone()

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-7):
        self.optimizer, self.warmup = optimizer, warmup_epochs
        self.total, self.min_lr = total_epochs, min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
    def step(self, epoch):
        if epoch < self.warmup:
            p = epoch / max(1, self.warmup)
            for pg, blr in zip(self.optimizer.param_groups, self.base_lrs): pg['lr'] = blr * p
        else:
            p = (epoch - self.warmup) / max(1, self.total - self.warmup)
            for pg, blr in zip(self.optimizer.param_groups, self.base_lrs):
                pg['lr'] = self.min_lr + (blr - self.min_lr) * 0.5 * (1 + math.cos(math.pi * p))

print(f'Model + Dataset + FocalLoss + EMA hazir')

In [ ]:
#@title 4 - 3 FAZLI EGITIM + SWA + R-DROP

# ===== NORMALIZATION =====
print('Normalization hesaplaniyor...')
all_feats = []
for i in range(0, len(X_train), 500):
    batch = X_train[i:i+500]
    for seq in batch:
        seq_p = pad_or_truncate(seq, cfg.SEQ_LENGTH)
        feat = build_features(seq_p, cfg)
        all_feats.append(feat)
all_feats = np.concatenate(all_feats, axis=0)
norm_mean = all_feats.mean(axis=0)
norm_std = np.maximum(all_feats.std(axis=0), 1e-6)
print(f'  Feature shape: {norm_mean.shape}')
del all_feats; gc.collect()

# Datasets
train_dataset = AUTSLDataset(X_train, y_train, cfg, is_train=True)
val_dataset = AUTSLDataset(X_val, y_val, cfg, is_train=False)
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE*2, shuffle=False,
                        num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
norm_mean_t = torch.tensor(norm_mean, dtype=torch.float32).to(device)
norm_std_t = torch.tensor(norm_std, dtype=torch.float32).to(device)
class_weights_d = class_weights.to(device)
print(f'Device: {device}')

# ===== MODEL =====
model = SignTransformerPro(
    input_size=FEATURE_SIZE,
    d_model=cfg.D_MODEL, nhead=cfg.NHEAD,
    num_layers=cfg.NUM_LAYERS, num_classes=cfg.NUM_CLASSES,
    dropout=cfg.DROPOUT
).to(device)

# Load pretrained (partial - skip input_conv due to size change)
pretrained_path = os.path.join(cfg.PROJE_DIR, 'Models_v3_80plus', 'best_model.pt')
if os.path.exists(pretrained_path):
    state = torch.load(pretrained_path, map_location=device, weights_only=True)
    pretrained_sd = state.get('model_state_dict', state)
    print(f'Pretrained val_acc: {state.get("val_acc", "?")}')
    model_sd = model.state_dict()
    loaded, skipped = 0, 0
    for k, v in pretrained_sd.items():
        if k in model_sd and model_sd[k].shape == v.shape:
            model_sd[k] = v; loaded += 1
        else:
            skipped += 1
    model.load_state_dict(model_sd)
    print(f'  {loaded} katman yuklendi, {skipped} atlanildi')
else:
    print(f'UYARI: Pretrained bulunamadi: {pretrained_path}')

total_params = sum(p.numel() for p in model.parameters())
print(f'Toplam parametre: {total_params:,}')

# ===== FREEZE HELPERS =====
def freeze_all(m):
    for p in m.parameters(): p.requires_grad = False
def unfreeze_all(m):
    for p in m.parameters(): p.requires_grad = True

def unfreeze_classifier_only(m):
    freeze_all(m)
    for p in m.classifier.parameters(): p.requires_grad = True
    for p in m.pool_heads.parameters(): p.requires_grad = True
    for p in m.input_conv.parameters(): p.requires_grad = True  # Yeni katman
    m.cls_token.requires_grad = True

def unfreeze_partial(m):
    freeze_all(m)
    for p in m.classifier.parameters(): p.requires_grad = True
    for p in m.pool_heads.parameters(): p.requires_grad = True
    for p in m.input_conv.parameters(): p.requires_grad = True
    for p in m.conv_block.parameters(): p.requires_grad = True
    for p in m.pos_encoder.parameters(): p.requires_grad = True
    m.cls_token.requires_grad = True
    n_layers = len(m.transformer.layers)
    for i in range(max(0, n_layers-3), n_layers):
        for p in m.transformer.layers[i].parameters(): p.requires_grad = True

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

# ===== VALIDATION =====
def validate(model, loader, device, nm, ns):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx = (bx.to(device) - nm) / ns
            by = by.to(device)
            with torch.amp.autocast('cuda'):
                out = model(bx)
            correct += (out.argmax(1) == by).sum().item()
            total += by.size(0)
    return 100 * correct / total

# ===== TRAIN ONE EPOCH (with R-Drop) =====
def train_epoch(model, loader, optimizer, criterion, scaler, device, nm, ns,
                epoch, total_epochs, use_rdrop=False, rdrop_alpha=0.5):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{total_epochs}')
    for bx, by in pbar:
        bx = (bx.to(device) - nm) / ns
        by = by.to(device)
        with torch.amp.autocast('cuda'):
            out1 = model(bx)
            loss = criterion(out1, by)
            # R-Drop: second forward pass, minimize KL between two outputs
            if use_rdrop and rdrop_alpha > 0:
                out2 = model(bx)
                loss2 = criterion(out2, by)
                p1 = F.log_softmax(out1, dim=1)
                p2 = F.log_softmax(out2, dim=1)
                q1 = F.softmax(out1.detach(), dim=1)
                q2 = F.softmax(out2.detach(), dim=1)
                kl = 0.5 * (F.kl_div(p1, q2, reduction='batchmean') +
                            F.kl_div(p2, q1, reduction='batchmean'))
                loss = 0.5 * (loss + loss2) + rdrop_alpha * kl
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * by.size(0)
        correct += (out1.argmax(1) == by).sum().item()
        total += by.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{100*correct/total:.1f}%')
    return total_loss / total, 100 * correct / total

# ===== SWA STATE TRACKING =====
swa_state = None
swa_n = 0

def update_swa(model):
    global swa_state, swa_n
    if swa_state is None:
        swa_state = {k: v.clone() for k, v in model.state_dict().items()}
        swa_n = 1
    else:
        for k in swa_state:
            swa_state[k] = (swa_state[k] * swa_n + model.state_dict()[k]) / (swa_n + 1)
        swa_n += 1

def apply_swa(model):
    if swa_state is not None:
        model.load_state_dict(swa_state)
    return model

# ===== MAIN TRAINING =====
criterion = FocalLoss(gamma=cfg.FOCAL_GAMMA, weight=class_weights_d,
                      label_smoothing=cfg.LABEL_SMOOTHING)
scaler = torch.amp.GradScaler('cuda')

best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'lr': [], 'phase': []}
global_epoch = 0

# Resume from checkpoint
ckpt_path = os.path.join(cfg.SAVE_DIR, 'checkpoint.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    ckpt_acc = ckpt.get('val_acc', 0)
    if ckpt_acc >= 50.0:
        model.load_state_dict(ckpt['model_state_dict'])
        global_epoch = ckpt['epoch'] + 1
        best_val_acc = ckpt.get('best_val_acc', ckpt_acc)
        if 'history' in ckpt: history = ckpt['history']
        if 'swa_state' in ckpt:
            swa_state = ckpt['swa_state']; swa_n = ckpt.get('swa_n', 1)
        print(f'Checkpoint yuklendi: epoch {global_epoch}, best={best_val_acc:.2f}%')
    else:
        os.remove(ckpt_path)
        print(f'Bozuk checkpoint silindi (acc={ckpt_acc:.1f}%)')

# ====== PHASE 1: Classifier + input_conv (yeni katmanlar) ======
p1_end = cfg.PHASE1_EPOCHS
if global_epoch < p1_end:
    print(f'\n{"="*60}')
    print(f'FAZ 1: Classifier + input_conv (epoch {global_epoch+1}-{p1_end})')
    print(f'{"="*60}')
    unfreeze_classifier_only(model)
    print(f'Trainable: {count_trainable(model):,}')
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=cfg.LR * 10, weight_decay=cfg.WEIGHT_DECAY)
    ema = EMAModel(model, decay=cfg.EMA_DECAY)
    for epoch in range(global_epoch, p1_end):
        t_loss, t_acc = train_epoch(model, train_loader, opt, criterion, scaler,
                                    device, norm_mean_t, norm_std_t, epoch, cfg.EPOCHS)
        ema.update(model)
        v_acc = validate(model, val_loader, device, norm_mean_t, norm_std_t)
        history['train_loss'].append(t_loss); history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc); history['lr'].append(opt.param_groups[0]['lr'])
        history['phase'].append(1)
        print(f'  [F1] train={t_acc:.1f}% val={v_acc:.2f}%')
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save({'model_state_dict': model.state_dict(), 'val_acc': v_acc, 'epoch': epoch},
                       os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
            print(f'  NEW BEST: {v_acc:.2f}%')
        global_epoch = epoch + 1

# ====== PHASE 2: Partial unfreeze (+son 3 transformer layer) ======
p2_end = cfg.PHASE1_EPOCHS + cfg.PHASE2_EPOCHS
if global_epoch < p2_end:
    print(f'\n{"="*60}')
    print(f'FAZ 2: Son 3 transformer + classifier (epoch {global_epoch+1}-{p2_end})')
    print(f'{"="*60}')
    unfreeze_partial(model)
    print(f'Trainable: {count_trainable(model):,}')
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=cfg.LR * 3, weight_decay=cfg.WEIGHT_DECAY)
    sched = WarmupCosineScheduler(opt, 3, p2_end - global_epoch)
    ema = EMAModel(model, decay=cfg.EMA_DECAY)
    patience = 0
    p2_start = global_epoch
    for epoch in range(global_epoch, p2_end):
        sched.step(epoch - p2_start)
        t_loss, t_acc = train_epoch(model, train_loader, opt, criterion, scaler,
                                    device, norm_mean_t, norm_std_t, epoch, cfg.EPOCHS,
                                    use_rdrop=True, rdrop_alpha=cfg.RDROP_ALPHA * 0.5)
        ema.update(model)
        v_acc = validate(model, val_loader, device, norm_mean_t, norm_std_t)
        history['train_loss'].append(t_loss); history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc); history['lr'].append(opt.param_groups[0]['lr'])
        history['phase'].append(2)
        print(f'  [F2] train={t_acc:.1f}% val={v_acc:.2f}% lr={opt.param_groups[0]["lr"]:.2e}')
        if v_acc > best_val_acc:
            best_val_acc = v_acc; patience = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_acc': v_acc, 'epoch': epoch},
                       os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
            print(f'  NEW BEST: {v_acc:.2f}%')
        else: patience += 1
        if (epoch+1) % 5 == 0:
            torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch,
                        'val_acc': v_acc, 'best_val_acc': best_val_acc, 'history': history},
                       ckpt_path)
        global_epoch = epoch + 1

# ====== PHASE 3: Full fine-tuning + SWA ======
if global_epoch < cfg.EPOCHS:
    print(f'\n{"="*60}')
    print(f'FAZ 3: Full fine-tuning + SWA (epoch {global_epoch+1}-{cfg.EPOCHS})')
    print(f'{"="*60}')
    unfreeze_all(model)
    print(f'Trainable: {count_trainable(model):,}')
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    remaining = cfg.EPOCHS - global_epoch
    sched = WarmupCosineScheduler(opt, 3, remaining)
    ema = EMAModel(model, decay=cfg.EMA_DECAY)
    patience = 0
    p3_start = global_epoch
    for epoch in range(global_epoch, cfg.EPOCHS):
        sched.step(epoch - p3_start)
        # SWA mode: Fixed LR after SWA_START_EPOCH
        if epoch >= cfg.SWA_START_EPOCH:
            for pg in opt.param_groups: pg['lr'] = cfg.SWA_LR
        t_loss, t_acc = train_epoch(model, train_loader, opt, criterion, scaler,
                                    device, norm_mean_t, norm_std_t, epoch, cfg.EPOCHS,
                                    use_rdrop=True, rdrop_alpha=cfg.RDROP_ALPHA)
        ema.update(model)
        # SWA accumulation
        if epoch >= cfg.SWA_START_EPOCH:
            eb_swa = ema.apply(model)
            update_swa(model)
            ema.restore(model, eb_swa)
        v_acc = validate(model, val_loader, device, norm_mean_t, norm_std_t)
        history['train_loss'].append(t_loss); history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc); history['lr'].append(opt.param_groups[0]['lr'])
        history['phase'].append(3)
        swa_info = f' swa_n={swa_n}' if epoch >= cfg.SWA_START_EPOCH else ''
        print(f'  [F3] train={t_acc:.1f}% val={v_acc:.2f}% lr={opt.param_groups[0]["lr"]:.2e}{swa_info}')
        if v_acc > best_val_acc:
            best_val_acc = v_acc; patience = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_acc': v_acc, 'epoch': epoch},
                       os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
            print(f'  NEW BEST: {v_acc:.2f}%')
        else: patience += 1
        if (epoch+1) % 5 == 0:
            save_d = {'model_state_dict': model.state_dict(), 'epoch': epoch,
                       'val_acc': v_acc, 'best_val_acc': best_val_acc, 'history': history}
            if swa_state: save_d['swa_state'] = swa_state; save_d['swa_n'] = swa_n
            torch.save(save_d, ckpt_path)
        if patience >= cfg.PATIENCE:
            print(f'  Early stopping ({cfg.PATIENCE} epoch iyilesme yok)')
            break
        global_epoch = epoch + 1

# SWA Final model
if swa_state is not None and swa_n > 1:
    print(f'\nSWA sonuclari ({swa_n} model ortalamasi)...')
    swa_model = SignTransformerPro(
        input_size=FEATURE_SIZE, d_model=cfg.D_MODEL, nhead=cfg.NHEAD,
        num_layers=cfg.NUM_LAYERS, num_classes=cfg.NUM_CLASSES, dropout=cfg.DROPOUT
    ).to(device)
    swa_model.load_state_dict(swa_state)
    # BN update with training data
    swa_model.train()
    with torch.no_grad():
        for i, (bx, by) in enumerate(train_loader):
            bx = (bx.to(device) - norm_mean_t) / norm_std_t
            swa_model(bx)
            if i >= 50: break  # 50 batch yeterli
    swa_acc = validate(swa_model, val_loader, device, norm_mean_t, norm_std_t)
    print(f'  SWA val_acc: {swa_acc:.2f}% (vs best EMA: {best_val_acc:.2f}%)')
    if swa_acc > best_val_acc:
        best_val_acc = swa_acc
        torch.save({'model_state_dict': swa_model.state_dict(), 'val_acc': swa_acc, 'epoch': -1, 'type': 'swa'},
                   os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
        print(f'  SWA model kaydedildi (YENI BEST!)')
    torch.save({'model_state_dict': swa_model.state_dict(), 'val_acc': swa_acc, 'type': 'swa'},
               os.path.join(cfg.SAVE_DIR, 'swa_model.pt'))
    del swa_model

print(f'\nEgitim tamamlandi! Best val_acc: {best_val_acc:.2f}%')

In [ ]:
#@title 5 - TTA TEST + ANALIZ + KAYIT

print('='*60)
print('TEST-TIME AUGMENTATION (TTA) DEGERLENDIRME')
print('='*60)

# Load best model
best_path = os.path.join(cfg.SAVE_DIR, 'best_model.pt')
best_state = torch.load(best_path, map_location=device, weights_only=True)
if 'model_state_dict' in best_state:
    model.load_state_dict(best_state['model_state_dict'])
else:
    model.load_state_dict(best_state)
print(f'Best model yuklendi (val_acc: {best_state.get("val_acc", "?")})')

# ===== TTA: 7 variants =====
def tta_predict(model, seq_raw, cfg, device, nm, ns, n_variants=7):
    """Predict with Test-Time Augmentation.
    Variants: original, mirror, +noise x2, speed+/speed-, temporal shift+/shift-"""
    model.eval()
    all_probs = []
    
    variants = [seq_raw.copy()]  # 1. Original
    
    # 2. Mirror
    m = seq_raw.copy()
    for i in range(0, min(225, m.shape[1]), 3): m[:, i] = 1.0 - m[:, i]
    if m.shape[1] >= 225:
        lh, rh = m[:, 99:162].copy(), m[:, 162:225].copy()
        m[:, 99:162], m[:, 162:225] = rh, lh
    variants.append(m)
    
    # 3-4. Small noise
    for _ in range(2):
        variants.append(seq_raw + np.random.randn(*seq_raw.shape).astype(np.float32) * 0.002)
    
    # 5. Slightly faster
    T = len(seq_raw)
    new_T = max(10, int(T * 0.95))
    idx = np.clip(np.linspace(0, T-1, new_T).astype(int), 0, T-1)
    variants.append(seq_raw[idx])
    
    # 6. Slightly slower
    new_T = min(T+5, int(T * 1.05))
    idx = np.clip(np.linspace(0, T-1, new_T).astype(int), 0, T-1)
    variants.append(seq_raw[idx])
    
    # 7. Temporal shift
    variants.append(np.roll(seq_raw, 1, axis=0))
    
    with torch.no_grad():
        for var in variants[:n_variants]:
            var_p = pad_or_truncate(var, cfg.SEQ_LENGTH)
            feat = build_features(var_p, cfg)
            x = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(device)
            x = (x - nm) / ns
            with torch.amp.autocast('cuda'):
                out = model(x)
            all_probs.append(F.softmax(out, dim=1))
    
    # Weighted average (original gets 2x weight)
    weights = [2.0] + [1.0] * (len(all_probs) - 1)
    total_w = sum(weights)
    avg_prob = sum(w * p for w, p in zip(weights, all_probs)) / total_w
    return avg_prob

# Test with TTA
test_correct_tta, test_correct_no_tta, test_total = 0, 0, 0
all_preds, all_labels, all_probs_list = [], [], []

# Also measure no-TTA for comparison
test_dataset = AUTSLDataset(X_test, y_test, cfg, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE*2, shuffle=False,
                          num_workers=2, pin_memory=True)

# No-TTA baseline
model.eval()
no_tta_correct, no_tta_total = 0, 0
with torch.no_grad():
    for bx, by in tqdm(test_loader, desc='Test (no TTA)'):
        bx = (bx.to(device) - norm_mean_t) / norm_std_t
        by = by.to(device)
        with torch.amp.autocast('cuda'):
            out = model(bx)
        no_tta_correct += (out.argmax(1) == by).sum().item()
        no_tta_total += by.size(0)
no_tta_acc = 100 * no_tta_correct / no_tta_total
print(f'Test (no TTA): {no_tta_acc:.2f}%')

# TTA inference (sample by sample)
print(f'TTA inference ({cfg.TTA_VARIANTS} variant)...')
for i in tqdm(range(len(X_test)), desc='TTA Test'):
    seq_raw = X_test[i].copy()
    label = y_test[i]
    
    avg_prob = tta_predict(model, seq_raw, cfg, device, norm_mean_t, norm_std_t, cfg.TTA_VARIANTS)
    pred = avg_prob.argmax(1).item()
    
    all_preds.append(pred)
    all_labels.append(label)
    all_probs_list.append(avg_prob.cpu().numpy()[0])
    if pred == label: test_correct_tta += 1
    test_total += 1

tta_acc = 100 * test_correct_tta / test_total
print(f'\nTest (TTA {cfg.TTA_VARIANTS}x): {tta_acc:.2f}% (+{tta_acc - no_tta_acc:.2f}% vs no-TTA)')

# Save normalization + label map
np.save(os.path.join(cfg.SAVE_DIR, 'norm_mean.npy'), norm_mean)
np.save(os.path.join(cfg.SAVE_DIR, 'norm_std.npy'), norm_std)
label_map = {str(i): class_names.get(i, f'class_{i}') for i in range(cfg.NUM_CLASSES)}
with open(os.path.join(cfg.SAVE_DIR, 'label_map.json'), 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

# Feature config (needed for inference later)
feat_config = {
    'feature_size': FEATURE_SIZE,
    'use_velocity': cfg.USE_VELOCITY,
    'use_acceleration': cfg.USE_ACCELERATION,
    'use_distances': cfg.USE_DISTANCES,
    'distance_pairs': DISTANCE_PAIRS,
    'raw_input': cfg.RAW_INPUT,
    'seq_length': cfg.SEQ_LENGTH,
    'd_model': cfg.D_MODEL,
    'nhead': cfg.NHEAD,
    'num_layers': cfg.NUM_LAYERS,
    'dropout': cfg.DROPOUT,
    'best_val_acc': best_val_acc,
    'test_acc_no_tta': no_tta_acc,
    'test_acc_tta': tta_acc
}
with open(os.path.join(cfg.SAVE_DIR, 'feature_config.json'), 'w') as f:
    json.dump(feat_config, f, indent=2, default=str)
print('Config + norm + label_map kaydedildi')

# ===== Training curves =====
if len(history['val_acc']) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    ep_range = range(1, len(history['val_acc'])+1)
    
    axes[0].plot(ep_range, history['train_loss'], 'b-', linewidth=1.5)
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True)
    
    axes[1].plot(ep_range, history['val_acc'], 'r-', linewidth=2, label='Val')
    axes[1].plot(ep_range, history['train_acc'], 'b--', linewidth=1, alpha=0.5, label='Train')
    axes[1].axhline(y=best_val_acc, color='g', linestyle=':', label=f'Best: {best_val_acc:.2f}%')
    # Phase boundaries
    axes[1].axvline(x=cfg.PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='Phase 2')
    axes[1].axvline(x=cfg.PHASE1_EPOCHS+cfg.PHASE2_EPOCHS, color='gray', linestyle='-.', alpha=0.5, label='Phase 3')
    if cfg.SWA_START_EPOCH < len(history['val_acc']):
        axes[1].axvline(x=cfg.SWA_START_EPOCH, color='orange', linestyle='--', alpha=0.5, label='SWA Start')
    axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch')
    axes[1].legend(fontsize=8); axes[1].grid(True)
    
    axes[2].plot(ep_range, history['lr'], 'g-', linewidth=1.5)
    axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
    axes[2].set_yscale('log'); axes[2].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.SAVE_DIR, 'training_curves.png'), dpi=150)
    plt.show()

# ===== Per-class analysis =====
all_preds_np = np.array(all_preds)
all_labels_np = np.array(all_labels)
all_probs_np = np.array(all_probs_list)

per_class_correct, per_class_total = {}, {}
for p, l in zip(all_preds_np, all_labels_np):
    per_class_total[l] = per_class_total.get(l, 0) + 1
    if p == l: per_class_correct[l] = per_class_correct.get(l, 0) + 1

worst = []
for cid in per_class_total:
    acc = 100 * per_class_correct.get(cid, 0) / per_class_total[cid]
    worst.append((cid, class_names.get(cid, f'class_{cid}'), acc, per_class_total[cid]))
worst.sort(key=lambda x: x[2])

print(f'\nEn dusuk 20 sinif:')
for cid, name, acc, n in worst[:20]:
    print(f'  {cid:>4} {name:<30} {acc:>6.1f}% (n={n})')

# Top-5
all_probs_t = torch.tensor(all_probs_np)
all_labels_t = torch.tensor(all_labels_np)
top5 = all_probs_t.topk(5, dim=1).indices
top5_correct = sum(1 for i in range(len(all_labels_t)) if all_labels_t[i] in top5[i])
top5_acc = 100 * top5_correct / len(all_labels_t)

correct_mask = all_preds_np == all_labels_np
correct_conf = all_probs_np[np.arange(len(all_preds_np)), all_preds_np][correct_mask]
wrong_conf = all_probs_np[np.arange(len(all_preds_np)), all_preds_np][~correct_mask]

# Sinif acc dagilimi
class_accs = [100 * per_class_correct.get(cid, 0) / per_class_total[cid] for cid in per_class_total]
above90 = sum(1 for a in class_accs if a >= 90)
above80 = sum(1 for a in class_accs if a >= 80)
below50 = sum(1 for a in class_accs if a < 50)

print(f'\n{"="*60}')
print(f'SONUC OZETI')
print(f'{"="*60}')
print(f'  Best Val:      {best_val_acc:.2f}%')
print(f'  Test (no TTA): {no_tta_acc:.2f}%')
print(f'  Test (TTA):    {tta_acc:.2f}%')
print(f'  Test Top-5:    {top5_acc:.2f}%')
print(f'  Dogru conf:    {correct_conf.mean():.4f}')
if len(wrong_conf) > 0:
    print(f'  Yanlis conf:   {wrong_conf.mean():.4f}')
print(f'  Sinif acc >= 90%: {above90}/{len(class_accs)}')
print(f'  Sinif acc >= 80%: {above80}/{len(class_accs)}')
print(f'  Sinif acc < 50%:  {below50}/{len(class_accs)}')
print(f'  Epochs:        {len(history["val_acc"])}')

print(f'\nKaydedilen dosyalar ({cfg.SAVE_DIR}):')
for f in sorted(os.listdir(cfg.SAVE_DIR)):
    fp = os.path.join(cfg.SAVE_DIR, f)
    sz = os.path.getsize(fp) / (1024*1024)
    print(f'  {f} ({sz:.1f} MB)')
print('\nTamamlandi!')